# LifeMate — Huấn luyện lại Qwen3.5-4B LoRA (v2) trên Google Colab G4

Notebook này train lại adapter `llm-lora-p1` từ bộ dữ liệu mới **`train_sft`** (gồm toàn bộ TRAIN + `recovery_v2` sửa lỗi tìm rỗng / thiếu mã + `synth_v2` sửa luồng `add_to_cart -> checkout` & giữ tỷ lệ hỏi lại).

### Cách dùng nhanh (3 bước)
1. Chọn **Runtime → Change runtime type → G4 GPU** (96 GB VRAM; cũng chạy được trên A100 hoặc L4).
2. Kiểm tra `DATASET_SPLIT` ở **Cell 4**:
   - `"v2_full"` *(mặc định)*: train toàn bộ **5,166 mẫu** (1,545 câu TRAIN gốc + bổ sung v2) để lấy model nộp Public/Docker.
   - `"v2_eval"`: train **4,065 mẫu** (bỏ 200 câu Val) nếu bạn muốn tự chấm lại điểm Val trước.
3. Chạy lần lượt từ **Cell 2 → Cell 6**. Checkpoint và adapter cuối sẽ tự động lưu vào Google Drive tại `Colab_Upload/trained_outputs/lifemate-v2/`.

In [ ]:
# Cell 2: Kiểm tra GPU, mount Google Drive, tìm Colab_Upload và đồng bộ code
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

print("=== GPU hiện tại ===")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=False)

MYDRIVE = Path("/content/drive/MyDrive")
PREFERRED_ROOTS = [
    MYDRIVE / "Colab_Upload",
    MYDRIVE / "Lifemate",
]

def is_upload_root(p: Path) -> bool:
    return (p / "Combined/pyproject.toml").is_file() and (p / "train_sft/v2_full/train.jsonl").is_file()

DRIVE_ROOT = next((p for p in PREFERRED_ROOTS if is_upload_root(p)), None)
if DRIVE_ROOT is None:
    for p in MYDRIVE.rglob("pyproject.toml"):
        cand = p.parent.parent
        if is_upload_root(cand):
            DRIVE_ROOT = cand
            break

assert DRIVE_ROOT is not None, (
    "Không tìm thấy thư mục Colab_Upload chứa Combined/ và train_sft/v2_full/train.jsonl trong MyDrive."
)

WORK_DIR = Path("/content/lifemate_train")
STAGE_SFT_DIR = WORK_DIR / "train_sft"
DRIVE_OUT_DIR = DRIVE_ROOT / "trained_outputs/lifemate-v2"
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DRIVE_ROOT    = {DRIVE_ROOT}")
print(f"WORK_DIR      = {WORK_DIR}")
print(f"DRIVE_OUT_DIR = {DRIVE_OUT_DIR}")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(
    DRIVE_ROOT / "Combined",
    WORK_DIR,
    ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", "models", "outputs"),
)
shutil.copytree(DRIVE_ROOT / "train_sft", STAGE_SFT_DIR)

# Kiểm tra xem có copy kèm bộ train gốc hay không (tùy chọn)
DRIVE_DATA_KIT = DRIVE_ROOT / "train+public_test"
HAS_RAW_TRAIN = (DRIVE_DATA_KIT / "train/train.jsonl").is_file()
print(f"Đã copy code & train_sft vào {WORK_DIR}. Có sẵn bộ train/ gốc trên Drive: {HAS_RAW_TRAIN}")

In [ ]:
# Cell 3: Cài đặt thư viện huấn luyện (Unsloth + PEFT + TRL + Transformers) và kiểm tra CUDA/BF16
import importlib.metadata
import subprocess
import sys

def _run_pip(args: list[str]) -> None:
    proc = subprocess.Popen(
        [sys.executable, "-m", "pip", *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, proc.args)

need_unsloth = False
try:
    import unsloth  # noqa: F401
except Exception:
    need_unsloth = True

if need_unsloth:
    print("Đang cài đặt unsloth & các gói SFT (giữ nguyên PyTorch/CUDA của Colab)...")
    _run_pip([
        "install", "--disable-pip-version-check",
        "unsloth", "unsloth_zoo", "trl", "peft", "accelerate", "datasets", "bitsandbytes", "huggingface_hub",
    ])
else:
    print("unsloth đã cài sẵn.")

_run_pip(["install", "--disable-pip-version-check", "-q", "--no-deps", "--no-build-isolation", "-e", str(WORK_DIR)])

import torch
from unsloth import FastModel
import transformers, peft, trl, unsloth
sys.path.insert(0, str(WORK_DIR / "src"))
import lifemate.agent

assert torch.cuda.is_available(), "Không thấy CUDA GPU!"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(
    f"OK | GPU={gpu_name} ({vram_gb:.1f} GB) | bf16={torch.cuda.is_bf16_supported()} "
    f"| torch={torch.__version__} | transformers={transformers.__version__} "
    f"| trl={trl.__version__} | peft={peft.__version__} | unsloth={unsloth.__version__}"
)

In [ ]:
# Cell 4: Cấu hình tập dữ liệu & Tải base model Qwen/Qwen3.5-4B
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path
from huggingface_hub import snapshot_download

# --- TÙY CHỌN ---
DATASET_SPLIT = "v2_full"       # "v2_full" (5,166 mẫu, toàn bộ TRAIN) hoặc "v2_eval" (4,065 mẫu, bỏ 200 câu Val)
REFRESH_REAL_ASR = False        # Đổi thành True NẾU bạn có copy train/ (794 MB) lên Drive và muốn tự chạy Gipformer ASR thật
# ----------------

os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)

BASE_MODEL_DIR = WORK_DIR / "models/qwen3.5-4b"
BASE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Đang tải/kiểm tra Qwen/Qwen3.5-4B -> {BASE_MODEL_DIR}")
snapshot_download(
    repo_id="Qwen/Qwen3.5-4B",
    revision="851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a",
    local_dir=str(BASE_MODEL_DIR),
)

if REFRESH_REAL_ASR:
    assert HAS_RAW_TRAIN, "REFRESH_REAL_ASR=True nhưng chưa thấy train/train.jsonl trong Drive/Colab_Upload/train+public_test/"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sherpa-onnx==2.0.1", "soundfile==0.14.0"], check=True)
    orig_asr_dir = WORK_DIR / "models/gipformer1.5-68m-rnnt"
    snapshot_download(
        repo_id="g-group-ai-lab/gipformer1.5-68m-rnnt",
        revision="dd9227dcd8705c13f33bdbe59728d546ab94480f",
        local_dir=str(orig_asr_dir),
    )
    local_kit = WORK_DIR / "data/train+public_test"
    if not local_kit.exists():
        local_kit.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(DRIVE_DATA_KIT, local_kit)
    asr_out_dir = WORK_DIR / "outputs/asr/gipformer_train"
    if not (asr_out_dir / "asr_outputs.jsonl").is_file():
        subprocess.run([
            sys.executable, str(WORK_DIR / "scripts/infer/asr_cache.py"),
            "--split", "train", "--model-dir", str(orig_asr_dir),
            "--output-dir", str(asr_out_dir), "--workers", "4", "--threads", "2",
        ], check=True)
    subprocess.run([
        sys.executable, str(WORK_DIR / "scripts/data/build_v2_sft_data.py"),
        "--data-root", str(local_kit),
        "--asr-cache", str(asr_out_dir / "asr_outputs.jsonl"),
        "--out-dir", str(STAGE_SFT_DIR),
    ], check=True)

DATA_DIR = STAGE_SFT_DIR / DATASET_SPLIT
assert (DATA_DIR / "train.jsonl").is_file(), f"Không thấy {DATA_DIR / 'train.jsonl'}"
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
print("Dataset sẵn sàng:", json.dumps(manifest, ensure_ascii=False, indent=2))

In [ ]:
# Cell 5: Huấn luyện Qwen3.5-4B LoRA (bf16 + fuzzy-weight 0.3) trên GPU G4 và lưu về Google Drive
import os
import shutil
import subprocess
import sys
import torch
from pathlib import Path

EPOCHS = 2.0
LR = 1e-4
FUZZY_WEIGHT = 0.3
LORA_R = 16
LORA_ALPHA = 32
SAVE_STEPS = 150
RESUME_IF_EXISTS = True

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
# Trên GPU G4 (96 GB VRAM), tắt gradient checkpointing giúp train nhanh hơn hẳn; dưới 64 GB thì bật 'on' để an toàn VRAM
GRAD_CKPT = "off" if vram_gb >= 70 else "on"

OUT_RUN_DIR = DRIVE_OUT_DIR / f"runs/{DATASET_SPLIT}"
OUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", str(WORK_DIR / "scripts/train/train_agent_sft.py"),
    "--data-dir", str(DATA_DIR),
    "--out", str(OUT_RUN_DIR),
    "--model", str(BASE_MODEL_DIR),
    "--epochs", str(EPOCHS),
    "--lr", str(LR),
    "--r", str(LORA_R),
    "--alpha", str(LORA_ALPHA),
    "--grad-accum", "8",
    "--batch", "1",
    "--save-steps", str(SAVE_STEPS),
    "--fuzzy-weight", str(FUZZY_WEIGHT),
    "--bf16-base",
    "--grad-ckpt", GRAD_CKPT,
    "--seed", "42",
]
if RESUME_IF_EXISTS and any(OUT_RUN_DIR.glob("checkpoint-*")):
    cmd.append("--resume")

print("Bắt đầu train với lệnh:")
print(" ", " ".join(cmd))
print(f"(VRAM={vram_gb:.1f} GB -> grad_ckpt={GRAD_CKPT}; log sẽ in trực tiếp bên dưới)")

env = {**os.environ, "PYTHONUNBUFFERED": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}
proc = subprocess.Popen(cmd, cwd=str(WORK_DIR), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"Training lỗi với exit code {rc}")

ADAPTER_SRC = OUT_RUN_DIR / "adapter"
ADAPTER_DST = DRIVE_OUT_DIR / "llm-lora-p1"
assert (ADAPTER_SRC / "adapter_model.safetensors").is_file(), f"Không thấy adapter tại {ADAPTER_SRC}"
if ADAPTER_DST.exists():
    shutil.rmtree(ADAPTER_DST)
shutil.copytree(ADAPTER_SRC, ADAPTER_DST)
print(f"\nXONG! Adapter mới đã được lưu an toàn về Google Drive: {ADAPTER_DST}")

In [ ]:
# Cell 6: Đóng gói adapter mới + ASR để chạy ngay bằng colab_predict.ipynb
import shutil
from huggingface_hub import snapshot_download

RELEASE_V2_DIR = DRIVE_OUT_DIR / "release_v2"
RELEASE_V2_DIR.mkdir(parents=True, exist_ok=True)

# Tải ASR Gipformer final (chỉ ~70 MB) từ hugebenevolence/lifemate-final nếu chưa có
asr_dst = RELEASE_V2_DIR / "asr-gipformer"
if not (asr_dst / "encoder.onnx").is_file():
    print("Đang tải asr-gipformer từ hugebenevolence/lifemate-final...")
    snapshot_download(
        repo_id="hugebenevolence/lifemate-final",
        allow_patterns=["asr-gipformer/*", "llm-lora-e1/*", "SHA256SUMS"],
        local_dir=str(RELEASE_V2_DIR),
    )

lora_p1_dst = RELEASE_V2_DIR / "llm-lora-p1"
if lora_p1_dst.exists():
    shutil.rmtree(lora_p1_dst)
shutil.copytree(DRIVE_OUT_DIR / "llm-lora-p1", lora_p1_dst)

print("Đã đóng gói xong tại Google Drive:", RELEASE_V2_DIR)
print("Các thư mục bên trong:", sorted(p.name for p in RELEASE_V2_DIR.iterdir()))
print(
    "\nBước tiếp theo:\n"
    "1. Mở colab_predict.ipynb.\n"
    "2. Ở Cell 5, đặt CUSTOM_LORA_P1_DIR = DRIVE_ROOT / 'trained_outputs/lifemate-v2/llm-lora-p1'\n"
    "   (và khuyến nghị đặt USE_ENSEMBLE = False ở Cell 6 trước để test riêng model mới vừa nhanh vừa không bị synth cũ kéo điểm).\n"
    "3. Chạy Cell 6 -> Cell 9 để lấy submission.jsonl mới nộp Public Test!"
)